# Test whether a model generalizes

Can a forecast estimated from earlier bike rentals work on later hours? Use two simple rules, lock the better one on October, then evaluate the locked rule on November and December.

The target is the total number of Capital Bikeshare rentals in a recorded hour. Each rule returns a whole number of rentals. The same recorded hours, predictions, and errors appear in [Unit 4 of the ML Foundations lab](https://sharanharsoor.github.io/ml-learning-lab/ml-foundations/#unit-4).

Data are from Fanaee-T (2013), [UCI Bike Sharing](https://doi.org/10.24432/C5W894), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

## 1. Read and sort the data

Run each cell in order. This notebook uses Python's standard library. It downloads the official ZIP if no local copy is set.

If you have the official ZIP locally, change `local_archive` to its file path. The notebook also accepts the environment variable `BIKESHARE_ARCHIVE`. Nothing here needs a Google Drive mount or account credentials.

In [ ]:
import csv
from io import BytesIO, TextIOWrapper
import os
from pathlib import Path
from statistics import mean
from urllib.request import urlopen
from zipfile import ZipFile

source = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
local_archive = os.environ.get("BIKESHARE_ARCHIVE")

if local_archive:
    archive_data = Path(local_archive).read_bytes()
else:
    with urlopen(source, timeout=30) as response:
        archive_data = response.read()

with ZipFile(BytesIO(archive_data)) as archive:
    with archive.open("hour.csv") as stream:
        rows = list(csv.DictReader(TextIOWrapper(stream, encoding="utf-8")))

rows.sort(key=lambda row: (row["dteday"], int(row["hr"])))
assert len(rows) == 17379
print(len(rows), "recorded hours")

## 2. Give each period one job

- **Training** supplies labels for estimating the means in both candidate rules.
- **Validation** compares the candidate rules on later hours.
- **Test** checks the final locked rule at the end.

The dates use `YYYY-MM-DD` format, so sorting and comparing these strings matches calendar order. Each row stays in one period.

In [ ]:
validation_start = "2012-10-01"
test_start = "2012-11-01"

train = [row for row in rows if row["dteday"] < validation_start]
validation = [row for row in rows if validation_start <= row["dteday"] < test_start]
test = [row for row in rows if row["dteday"] >= test_start]

for role, part in [("Training", train), ("Validation", validation), ("Test", test)]:
    print(f'{role:10s} {part[0]["dteday"]} to {part[-1]["dteday"]} | {len(part):,} recorded hours')

assert (len(train), len(validation), len(test)) == (15211, 708, 1460)
assert train[-1]["dteday"] < validation[0]["dteday"] < test[0]["dteday"]

## 3. Fit two simple rules

The overall mean is a baseline. The calendar rule calculates a separate mean for each hour and workday group. This lets its prediction change with the time of day.

`fit_rules` reads the target column `cnt` from training rows. At forecast time, a candidate reads only the calendar fields it needs. `evaluate` opens the recorded labels to score predictions.

In [ ]:
def calendar_key(row):
    return row["hr"], row["workingday"]


def fit_rules(rows):
    groups = {}
    for row in rows:
        groups.setdefault(calendar_key(row), []).append(int(row["cnt"]))

    baseline = round(mean(int(row["cnt"]) for row in rows))
    calendar = {key: round(mean(counts)) for key, counts in groups.items()}
    return {
        "Overall mean": lambda row: baseline,
        "Calendar model": lambda row: calendar[calendar_key(row)],
    }


def evaluate(rules, rows):
    return {
        name: mean(abs(int(row["cnt"]) - predict(row)) for row in rows)
        for name, predict in rules.items()
    }

## 4. Use validation to choose a rule

Mean absolute error, or MAE, is the sum of absolute errors divided by the number of recorded hours. An error of 210 rentals counts as 210 whether we forecast too high or too low.

In [ ]:
candidates = fit_rules(train)
validation_errors = evaluate(candidates, validation)
chosen = min(validation_errors, key=validation_errors.get)

for name, mae in validation_errors.items():
    print(f"{name:15s} {round(mae * len(validation)):,} ÷ {len(validation):,} = {mae:.2f} rentals")
print("Lock the", chosen)
assert chosen == "Calendar model"

## 5. Refit through October and score the final test

October checked the rules before the choice. Once the rule is locked, both forecasts can use October as additional training history. Neither uses November or December to fit. We report both the locked rule and the baseline so the final MAE has a reference.

In [ ]:
final = fit_rules(train + validation)
test_errors = evaluate(
    {chosen: final[chosen], "Overall mean": final["Overall mean"]},
    test,
)

for name, mae in test_errors.items():
    print(f"{name:15s} {round(mae * len(test)):,} ÷ {len(test):,} = {mae:.2f} rentals")

focus_hour, = [row for row in test if (row["dteday"], row["hr"]) == ("2012-11-01", "8")]
forecast = final[chosen](focus_hour)
actual = int(focus_hour["cnt"])
print(f'November 1 at 08:00 | forecast {forecast} | actual {actual} | absolute error {abs(actual - forecast)}')
assert (forecast, actual, final["Overall mean"](focus_hour)) == (470, 680, 189)
assert abs(test_errors[chosen] - 63.51575342465753) < 1e-10
assert abs(test_errors["Overall mean"] - 140.08424657534246) < 1e-10

## 6. Spot a shortcut that leaks the answer

The data include casual and registered rental counts for the same hour. UCI defines total count as their sum. Using that sum would pass an accounting check after the hour. A forecast made at 7:59 a.m. would have to wait for those counts.

In [ ]:
casual = int(focus_hour["casual"])
registered = int(focus_hour["registered"])
target_from_after_the_hour = casual + registered
print(f"After-hour counts: {casual} + {registered} = {target_from_after_the_hour}")
print(f"Absolute error using information from after the hour: {abs(actual - target_from_after_the_hour)}")
assert (int(focus_hour["instant"]), casual, registered) == (15928, 12, 668)

## Try it yourself

1. Find the 09:00 record on November 1 in `test`. Predict with the locked calendar rule, then calculate its absolute error.
2. Explain why a random split of the historical hours would answer a different question from forecasting later hours.
3. Suppose the task changes to counting rentals at stations the model has never trained on. What should stay together when you split the station data?

Check after trying. The forecast is 238, the recorded count is 331, and the absolute error is 93 rentals. A chronological split simulates forecasting the future. Group all rows from a station together if the test must use unseen stations.